This notebook prepares the DAWN dataset for object detection experiments. It includes:
*   Data structuring
*   Train/validation/test split
*   Weather-specific subsets
*   YAML configuration generation

### Connexion to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import modules

In [2]:
import os
import random
import shutil
from pathlib import Path

### Delete generated folders and files if needed

In [ ]:
processed_path = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed")

if processed_path.exists():
    shutil.rmtree(processed_path)
    print("Processed folder deleted")

processed_path.mkdir(parents=True, exist_ok=True)
print("Processed folder recreated")

Processed folder deleted
Processed folder recreated


# **Raw dataset check**

### Verify DAWN dataset structure

In [ ]:
dataset_root = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/DAWN")

print("Dataset exists:", os.path.exists(dataset_root))
print("Contents:", os.listdir(dataset_root))

Dataset exists: True
Contents: ['Rain', 'Snow', 'Fog']


### Output folders for Yolo and Faster R-CNN

In [ ]:
output_root_yolo = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_yolo")
output_root_voc = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc")

print("Dataset root:", dataset_root)

Dataset root: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/raw/DAWN


# **YOLO dataset preparation**

### Create folders structure

In [ ]:
folders = [
    "images/train", "images/val", "images/test_rain", "images/test_snow", "images/test_fog",
    "labels/train", "labels/val", "labels/test_rain", "labels/test_snow", "labels/test_fog"
]

for folder in folders:
  (output_root_yolo / folder).mkdir(parents=True, exist_ok=True)
print("Folders created in:", output_root_yolo)

Folders created in: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_yolo


# **Faster R-CNN (VOC) dataset preparation**

### Create folders structure

In [ ]:
folders_voc = [
    "JPEGImages",
    "Annotations",
    "ImageSets/Main"
]

for folder in folders_voc:
   (output_root_voc / folder).mkdir(parents=True, exist_ok=True)

print("VOC folders created in:", output_root_voc)

VOC folders created in: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc


# **Set split parameters**

In [ ]:
random.seed(42)

val_ratio = 0.2
test_ratio = 0.2

weather_configs = {
    "Fog": ("Fog_YOLO_darknet", "Fog_PASCAL_VOC"),
    "Rain": ("Rain_YOLO_darknet", "Rain_PASCAL_VOC"),
    "Snow": ("Snow_YOLO_darknet", "Snow_PASCAL_VOC")
}

image_extensions = [".jpg", ".jpeg", ".png"]

### Split and copy image-label pairs

In [ ]:
summary = {}

splits_voc = {
    "train": [],
    "val": [],
    "test_fog": [],
    "test_rain": [],
    "test_snow": []
}

for weather_name, (yolo_folder, voc_folder) in weather_configs.items():
  print("Processing:", weather_name)

  weather_path = dataset_root /weather_name
  yolo_path = weather_path / yolo_folder
  voc_path = weather_path / voc_folder

  if not weather_path.exists():
    print("Weather folder not found:", weather_path)
    continue

  if not yolo_path.exists():
    print("Yolo folder not found:", yolo_path)
    continue

  if not voc_path.exists():
    print("VOC folder not found:", voc_path)
    continue

  samples = []

  for file in weather_path.iterdir():
    if file.suffix.lower() in image_extensions:
      image_file = file
      yolo_label = yolo_path / f"{file.stem}.txt"
      voc_label = voc_path / f"{file.stem}.xml"

      if yolo_label.exists() and voc_label.exists():
        samples.append((image_file, yolo_label, voc_label))

  print("Total pairs found:", len(samples))

  random.shuffle(samples)

  # Proper 60-20-20 split
  total = len(samples)

  test_size = int(total * test_ratio)
  val_size = int(total * val_ratio)
  train_size = total - test_size - val_size

  test_samples = samples[:test_size]
  val_samples = samples[test_size:test_size + val_size]
  train_samples = samples[test_size + val_size:]

  print("Train:", len(train_samples))
  print("Val:", len(val_samples))
  print("Test:", len(test_samples))

  #Test name
  test_folder = f"test_{weather_name.lower()}"

  #Train
  for image_file, yolo_label, voc_label in train_samples:
    shutil.copy2(image_file, output_root_yolo / "images/train" / image_file.name)
    shutil.copy2(yolo_label, output_root_yolo / "labels/train" / yolo_label.name)

    shutil.copy2(image_file, output_root_voc / "JPEGImages" / image_file.name)
    shutil.copy2(voc_label, output_root_voc / "Annotations" / voc_label.name)

    splits_voc["train"].append(image_file.stem)

  #Val
  for image_file, yolo_label, voc_label in val_samples:
    shutil.copy2(image_file, output_root_yolo / "images/val" / image_file.name)
    shutil.copy2(yolo_label, output_root_yolo / "labels/val" / yolo_label.name)

    shutil.copy2(image_file, output_root_voc / "JPEGImages" / image_file.name)
    shutil.copy2(voc_label, output_root_voc / "Annotations" / voc_label.name)

    splits_voc["val"].append(image_file.stem)

  #Test
  for image_file, yolo_label, voc_label in test_samples:
    shutil.copy2(image_file, output_root_yolo / f"images/{test_folder}" / image_file.name)
    shutil.copy2(yolo_label, output_root_yolo / f"labels/{test_folder}" / yolo_label.name)

    shutil.copy2(image_file, output_root_voc / "JPEGImages" / image_file.name)
    shutil.copy2(voc_label, output_root_voc / "Annotations" / voc_label.name)

    splits_voc[test_folder].append(image_file.stem)

  summary[weather_name] = {
      "total": len(samples),
      "train": len(train_samples),
      "val": len(val_samples),
      "test": len(test_samples)
  }

print("Summary:", summary)

Processing: Fog
Total pairs found: 300
Train: 180
Val: 60
Test: 60
Processing: Rain
Total pairs found: 200
Train: 120
Val: 40
Test: 40
Processing: Snow
Total pairs found: 204
Train: 124
Val: 40
Test: 40
Summary: {'Fog': {'total': 300, 'train': 180, 'val': 60, 'test': 60}, 'Rain': {'total': 200, 'train': 120, 'val': 40, 'test': 40}, 'Snow': {'total': 204, 'train': 124, 'val': 40, 'test': 40}}


# **Label YOLO**

### Check files without labels and remove them

In [ ]:
for split in ["train", "val", "test_fog", "test_rain", "test_snow"]:
    labels_dir = output_root_yolo / "labels" / split
    images_dir = output_root_yolo / "images" / split

    removed = 0

    for file_path in labels_dir.glob("*.txt"):
        with open(file_path, "r") as f:
            content = f.read().strip()

        if not content:
            image_stem = file_path.stem

            # remove empty label
            file_path.unlink()

            # remove matching image
            for ext in [".jpg", ".jpeg", ".png"]:
                img_path = images_dir / f"{image_stem}{ext}"
                if img_path.exists():
                    img_path.unlink()
                    break

            removed += 1

    print(f"{split}: removed {removed} empty labels and matching images")

train: removed 1 empty labels and matching images
val: removed 0 empty labels and matching images
test_fog: removed 0 empty labels and matching images
test_rain: removed 0 empty labels and matching images
test_snow: removed 0 empty labels and matching images


In [ ]:
empty_label_files_by_split = {}

for split in ["train", "val", "test_fog", "test_rain", "test_snow"]:
    labels_dir = output_root_yolo / "labels" / split
    empty_files = []

    for file_path in labels_dir.glob("*.txt"):
        with open(file_path, "r") as f:
            content = f.read().strip()

        if not content:
            empty_files.append(file_path.name)

    empty_label_files_by_split[split] = empty_files

print("Empty label files:")
for split, files in empty_label_files_by_split.items():
    print(split, ":", len(files))

Empty label files:
train : 0
val : 0
test_fog : 0
test_rain : 0
test_snow : 0


### Check labels' names

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

xml_class_counts = Counter()

annotations_dir = output_root_voc / "Annotations"

for xml_file in annotations_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip()
        xml_class_counts[class_name] += 1

print("All classes found in VOC annotations:")
for cls, count in sorted(xml_class_counts.items()):
    print(f"{cls}: {count}")

All classes found in VOC annotations:
bicycle: 17
bus: 106
car: 4706
motorcycle: 28
person: 303
train: 1
truck: 450


Note:  Just one train annotatin so i removed it to counteract the imbalances

In [ ]:
import xml.etree.ElementTree as ET

mapping = {}

for weather in ["Fog", "Rain", "Snow"]:
    yolo_dir = dataset_root / weather / weather_configs[weather][0]
    voc_dir = dataset_root / weather / weather_configs[weather][1]

    for yolo_file in yolo_dir.glob("*.txt"):
        name = yolo_file.stem
        xml_file = voc_dir / f"{name}.xml"

        if not xml_file.exists():
            continue

        with open(yolo_file, "r") as f:
            yolo_lines = [line.strip() for line in f if line.strip()]

        tree = ET.parse(xml_file)
        root = tree.getroot()
        xml_objects = root.findall("object")

        for yolo_line, obj in zip(yolo_lines, xml_objects):
            class_id = int(float(yolo_line.split()[0]))
            class_name = obj.find("name").text.strip()

            mapping.setdefault(class_id, set()).add(class_name)

print("YOLO ID → possible class names")
for k in sorted(mapping.keys()):
    print(k, "->", mapping[k])

YOLO ID → possible class names
1 -> {'person'}
2 -> {'bicycle'}
3 -> {'car'}
4 -> {'motorcycle'}
6 -> {'bus'}
7 -> {'train'}
8 -> {'truck'}


### Remap YOLO labels

In [ ]:
from types import new_class
old_to_new = {
    1: 0,  # person
    2: 1,  # bicycle
    3: 2,  # car
    4: 3,  # motorcycle
    6: 4,  # bus
    8: 5   # truck
}

splits = ["train", "val", "test_fog", "test_rain", "test_snow"]

for split in splits:
  labels_dir = output_root_yolo / "labels" / split

  for file_path in labels_dir.glob("*.txt"):
    new_lines = []

    with open(file_path, "r") as f:
      lines = [line.strip() for line in f if line.strip()]

    for line in lines:
      parts = line.split()

      old_class = int(float(parts[0]))

      if old_class not in old_to_new:
        continue

      new_class = old_to_new[old_class]
      coords = parts[1:]

      new_lines.append(" ".join([str(new_class)] + coords))
    with open(file_path, "w") as f:
      for line in new_lines:
        f.write(line + "\n")

  print(f"Remapped labels in {split} split")

Remapped labels in train split
Remapped labels in val split
Remapped labels in test_fog split
Remapped labels in test_rain split
Remapped labels in test_snow split


### Check mapping

In [ ]:
all_classes = set()

for split in splits:
    labels_dir = output_root_yolo / "labels" / split
    for file_path in labels_dir.glob("*.txt"):
        with open(file_path, "r") as f:
            for line in f:
                if line.strip():
                    all_classes.add(int(line.split()[0]))

print("Classes after remap:", all_classes)

Classes after remap: {0, 1, 2, 3, 4, 5}


### Check missing labels after remap

In [ ]:
empty_label_files_by_split = {}

for split in ["train", "val", "test_fog", "test_rain", "test_snow"]:
    labels_dir = output_root_yolo / "labels" / split
    empty_files = []

    for file_path in labels_dir.glob("*.txt"):
        with open(file_path, "r") as f:
            content = f.read().strip()

        if not content:
            empty_files.append(file_path.name)

    empty_label_files_by_split[split] = empty_files

print("Empty label files after remap:")
for split, files in empty_label_files_by_split.items():
    print(f"{split}: {len(files)}")
    if files:
        print("Examples:", files[:5])

Empty label files after remap:
train: 0
val: 0
test_fog: 0
test_rain: 0
test_snow: 0


### YAML Configuration files



In [ ]:
nc = 6

names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

yaml_content = f"""
path: {output_root_yolo}
train: images/train
val: images/val

nc: {nc}
names: {names}
"""

yaml_fog = f"""
path: {output_root_yolo}
train: images/train
val: images/test_fog

nc: {nc}
names: {names}
"""

yaml_rain = f"""
path: {output_root_yolo}
train: images/train
val: images/test_rain

nc: {nc}
names: {names}
"""

yaml_snow = f"""
path: {output_root_yolo}
train: images/train
val: images/test_snow

nc: {nc}
names: {names}
"""

with open(output_root_yolo / "dataset.yaml", "w") as f:
    f.write(yaml_content)

with open(output_root_yolo / "fog_only.yaml", "w") as f:
    f.write(yaml_fog)

with open(output_root_yolo / "rain_only.yaml", "w") as f:
    f.write(yaml_rain)

with open(output_root_yolo / "snow_only.yaml", "w") as f:
    f.write(yaml_snow)

print("YAML files created.")

YAML files created.


In [ ]:
with open(output_root_yolo / "dataset.yaml", "r") as f:
  print(f.read())


path: /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_yolo
train: images/train
val: images/val

nc: 6
names: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']



In [ ]:
for file_name in ["dataset.yaml", "fog_only.yaml", "rain_only.yaml", "snow_only.yaml"]:
    file_path = output_root_yolo / file_name
    print(file_name, "exists:", file_path.exists())

dataset.yaml exists: True
fog_only.yaml exists: True
rain_only.yaml exists: True
snow_only.yaml exists: True


# **Clean VOC Files**


*   Remove the image with train annotation
*   Remove empty XML files and matching images



In [ ]:
import xml.etree.ElementTree as ET

allowed_classes = {"person", "bicycle", "car", "motorcycle", "bus", "truck"}

removed_objects = 0
removed_files = []

annotations_dir = output_root_voc / "Annotations"
images_dir = output_root_voc / "JPEGImages"

for xml_file in annotations_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    objects = root.findall("object")
    kept_any = False

    for obj in objects:
        class_name = obj.find("name").text.strip()

        if class_name not in allowed_classes:
            root.remove(obj)
            removed_objects += 1
        else:
            kept_any = True

    if kept_any:
        tree.write(xml_file)
    else:
        removed_files.append(xml_file.stem)

# Remove empty XML files and matching images
for stem in removed_files:
    xml_path = annotations_dir / f"{stem}.xml"
    if xml_path.exists():
        xml_path.unlink()

    for ext in [".jpg", ".jpeg", ".png"]:
        img_path = images_dir / f"{stem}{ext}"
        if img_path.exists():
            img_path.unlink()
            break

print("Removed disallowed objects:", removed_objects)
print("Removed files with empty annotations:", len(removed_files))
if removed_files:
    print("Examples:", removed_files[:5])

Removed disallowed objects: 1
Removed files with empty annotations: 1
Examples: ['haze-012']


### Update splits_voc

In [ ]:
removed_files_set = set(removed_files)

for split in splits_voc:
    splits_voc[split] = [name for name in splits_voc[split] if name not in removed_files_set]

print("Updated splits_voc after removing empty VOC files.")

Updated splits_voc after removing empty VOC files.


# **Create train.txt and val.txt files for VOC**

In [ ]:
for split, names in splits_voc.items():
  file_path = output_root_voc / "ImageSets" / "Main" / f"{split}.txt"

  with open(file_path, "w") as f:
    for name in names:
      f.write(name + "\n")

print("VOC split files created")

VOC split files created


In [ ]:
split_files = ["train.txt", "val.txt", "test_fog.txt", "test_rain.txt", "test_snow.txt"]

for file_name in split_files:
    file_path = output_root_voc / "ImageSets" / "Main" / file_name
    print(f"{file_name} exists:", file_path.exists())

train.txt exists: True
val.txt exists: True
test_fog.txt exists: True
test_rain.txt exists: True
test_snow.txt exists: True


# **Integrity checks**

### Count images and labels for YOLO

In [ ]:
splits = ["train", "val", "test_fog", "test_rain", "test_snow"]

for split in splits:
    images = list((output_root_yolo / "images" / split).glob("*"))
    labels = list((output_root_yolo / "labels" / split).glob("*.txt"))

    print(f"\n {split} ")
    print("Images:", len(images))
    print("Labels:", len(labels))

    missing = []

    for img in images:
        label = output_root_yolo / "labels" / split / f"{img.stem}.txt"
        if not label.exists():
            missing.append(img.name)

    print("Missing labels:", len(missing))
    if missing:
        print("Examples:", missing[:5])


 train 
Images: 423
Labels: 423
Missing labels: 0

 val 
Images: 140
Labels: 140
Missing labels: 0

 test_fog 
Images: 60
Labels: 60
Missing labels: 0

 test_rain 
Images: 40
Labels: 40
Missing labels: 0

 test_snow 
Images: 40
Labels: 40
Missing labels: 0


### Count images and labels for VOC





In [ ]:
from pathlib import Path

# Count images and labels
voc_images = list((output_root_voc / "JPEGImages").glob("*"))
voc_annotations = list((output_root_voc / "Annotations").glob("*.xml"))

print("VOC images:", len(voc_images))
print("VOC annotations:", len(voc_annotations))

# Verify each image have annotation
missing_xml = []

for img_path in voc_images:
    xml_path = output_root_voc / "Annotations" / f"{img_path.stem}.xml"
    if not xml_path.exists():
        missing_xml.append(img_path.name)

print("Images without XML:", len(missing_xml))
if missing_xml:
    print("Examples:", missing_xml[:5])

# Verify splits
split_files = ["train", "val", "test_fog", "test_rain", "test_snow"]

for split in split_files:
    split_path = output_root_voc / "ImageSets" / "Main" / f"{split}.txt"

    print(f"\n {split} ")

    if not split_path.exists():
        print("Split file missing:", split_path)
        continue

    with open(split_path, "r") as f:
        names = [line.strip() for line in f if line.strip()]

    print("Entries:", len(names))

    missing_img = []
    missing_ann = []

    for name in names:
        # search jpg/png/jpeg
        img_exists = (
            (output_root_voc / "JPEGImages" / f"{name}.jpg").exists() or
            (output_root_voc / "JPEGImages" / f"{name}.jpeg").exists() or
            (output_root_voc / "JPEGImages" / f"{name}.png").exists()
        )

        ann_exists = (output_root_voc / "Annotations" / f"{name}.xml").exists()

        if not img_exists:
            missing_img.append(name)

        if not ann_exists:
            missing_ann.append(name)

    print("Missing images:", len(missing_img))
    if missing_img:
        print("Image examples:", missing_img[:5])

    print("Missing annotations:", len(missing_ann))
    if missing_ann:
        print("Annotation examples:", missing_ann[:5])

VOC images: 703
VOC annotations: 703
Images without XML: 0

 train 
Entries: 423
Missing images: 0
Missing annotations: 0

 val 
Entries: 140
Missing images: 0
Missing annotations: 0

 test_fog 
Entries: 60
Missing images: 0
Missing annotations: 0

 test_rain 
Entries: 40
Missing images: 0
Missing annotations: 0

 test_snow 
Entries: 40
Missing images: 0
Missing annotations: 0


In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

xml_class_counts = Counter()

annotations_dir = output_root_voc / "Annotations"

for xml_file in annotations_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip()
        xml_class_counts[class_name] += 1

print("Classes found in XML annotations:")

for cls, count in sorted(xml_class_counts.items()):
    print(f"{cls}: {count}")

Classes found in XML annotations:
bicycle: 17
bus: 106
car: 4706
motorcycle: 28
person: 303
truck: 450


### Count images and labels for YOLO

In [4]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/images")

for split in ["train", "val", "test_fog", "test_rain", "test_snow"]:
    n = len(list((base / split).glob("*.*")))
    print(split, n)

train 423
val 140
test_fog 60
test_rain 40
test_snow 40
